# Google Sheets Parser

Интерактивный парсер для работы с Google Sheets. Позволяет:
1. Получить список всех вкладок
2. Консолидировать данные со всех вкладок
3. Анализировать структуру данных
4. Сохранять в БД

In [3]:
# Импорты и настройка
import asyncio
import aiohttp
from typing import List, Dict, Any, Optional, Set
from dataclasses import dataclass
import json
import pandas as pd
import nest_asyncio

# Применяем nest_asyncio для работы с async в Jupyter
nest_asyncio.apply()

# Импорты из нашего приложения
from app.logger import logger
from app.config import settings

print("✅ Импорты загружены")

# Проверяем конфигурацию
print(f"Google Sheets API Key: {'✅ Установлен' if hasattr(settings, 'GOOGLE_SHEETS_API_KEY') and settings.GOOGLE_SHEETS_API_KEY else '❌ Не установлен'}")
print(f"Database URL: {'✅ Установлен' if hasattr(settings, 'DATABASE_URL') and settings.DATABASE_URL else '❌ Не установлен'}")

# Конфигурация
SPREADSHEET_ID = "1mh-hzIRyHz9D4_1nPa1CVQLr9RgSBpFD-vL_qzR5-ZA"
EXCLUDE_SHEETS = ["! Пояснения"]  # Вкладки для исключения

print(f"📊 ID таблицы: {SPREADSHEET_ID}")
print(f"🚫 Исключаем вкладки: {EXCLUDE_SHEETS}")

# Классы для работы с данными
@dataclass
class SheetInfo:
    """Информация о вкладке Google Sheets"""
    sheet_id: str
    title: str
    index: int
    grid_properties: Dict[str, Any]


@dataclass
class ColumnInfo:
    """Информация о колонке"""
    name: str
    sheet_title: str
    index: int
    sample_values: List[Any]


@dataclass
class ConsolidatedData:
    """Консолидированные данные"""
    columns: List[str]
    data: List[Dict[str, Any]]
    source_sheets: List[str]
    total_rows: int

print("✅ Классы данных определены")

✅ Импорты загружены
Google Sheets API Key: ✅ Установлен
Database URL: ✅ Установлен
📊 ID таблицы: 1mh-hzIRyHz9D4_1nPa1CVQLr9RgSBpFD-vL_qzR5-ZA
🚫 Исключаем вкладки: ['! Пояснения']
✅ Классы данных определены


In [4]:
# Основной класс парсера
class GoogleSheetsParser:
    """Парсер для Google Sheets"""
    
    def __init__(self, spreadsheet_id: str, api_key: Optional[str] = None):
        self.spreadsheet_id = spreadsheet_id
        self.api_key = api_key or settings.GOOGLE_SHEETS_API_KEY
        self.base_url = "https://sheets.googleapis.com/v4/spreadsheets"
        
        if not self.api_key:
            raise ValueError("Google Sheets API Key не указан. Добавьте GOOGLE_SHEETS_API_KEY в .env файл")
        
    async def get_sheets_list(self, exclude_sheets: Optional[List[str]] = None) -> List[SheetInfo]:
        """Получает список всех вкладок в Google Sheets"""
        if exclude_sheets is None:
            exclude_sheets = []
            
        url = f"{self.base_url}/{self.spreadsheet_id}"
        params = {'key': self.api_key}
            
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url, params=params) as response:
                    if response.status == 200:
                        data = await response.json()
                        sheets = []
                        
                        for sheet in data.get('sheets', []):
                            sheet_info = SheetInfo(
                                sheet_id=sheet['properties']['sheetId'],
                                title=sheet['properties']['title'],
                                index=sheet['properties']['index'],
                                grid_properties=sheet['properties'].get('gridProperties', {})
                            )
                            
                            # Исключаем вкладки по названию
                            if sheet_info.title not in exclude_sheets:
                                sheets.append(sheet_info)
                                # print(f"✅ Добавлена вкладка: {sheet_info.title}")
                            else:
                                print(f"🚫 Исключена вкладка: {sheet_info.title}")
                        
                        print(f"📊 Всего найдено вкладок: {len(data.get('sheets', []))}")
                        print(f"🔄 Вкладок для обработки: {len(sheets)}")
                        
                        return sheets
                    else:
                        error_text = await response.text()
                        print(f"❌ Ошибка получения списка вкладок: {response.status} - {error_text}")
                        return []
                        
        except Exception as e:
            print(f"❌ Ошибка при получении списка вкладок: {str(e)}")
            return []
    
    async def get_sheet_data(self, sheet_title: str, range_name: Optional[str] = None) -> List[List[Any]]:
        """Получает данные с конкретной вкладки"""
        if range_name is None:
            range_name = f"{sheet_title}!A:Z"  # По умолчанию берем все колонки
            
        url = f"{self.base_url}/{self.spreadsheet_id}/values/{range_name}"
        params = {'key': self.api_key}
            
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url, params=params) as response:
                    if response.status == 200:
                        data = await response.json()
                        values = data.get('values', [])
                        print(f"📥 Получено {len(values)} строк с вкладки {sheet_title}")
                        return values
                    else:
                        error_text = await response.text()
                        print(f"❌ Ошибка получения данных с вкладки {sheet_title}: {response.status} - {error_text}")
                        return []
                        
        except Exception as e:
            print(f"❌ Ошибка при получении данных с вкладки {sheet_title}: {str(e)}")
            return []


# Создаем экземпляр парсера
try:
    parser = GoogleSheetsParser(SPREADSHEET_ID)
    print("✅ Парсер инициализирован")
except Exception as e:
    print(f"❌ Ошибка инициализации: {e}")

✅ Парсер инициализирован


In [5]:
# Шаг 1: Получаем список всех вкладок
print("🔄 Шаг 1: Получаем список вкладок...")
sheets = await parser.get_sheets_list(EXCLUDE_SHEETS)

if sheets:
    print(f"\n📋 Список вкладок для обработки:")
    for i, sheet in enumerate(sheets, 1):
        print(f"  {i:2d}.\t{sheet.title}\t(ID: {sheet.sheet_id})")
else:
    print("❌ Не удалось получить список вкладок")

🔄 Шаг 1: Получаем список вкладок...
🚫 Исключена вкладка: ! Пояснения
📊 Всего найдено вкладок: 20
🔄 Вкладок для обработки: 19

📋 Список вкладок для обработки:
   1.	Июль 2025	(ID: 1046342236)
   2.	Июнь 2025	(ID: 2111477540)
   3.	Май 2025	(ID: 87923692)
   4.	Апрель 2025	(ID: 1992430774)
   5.	Март 2025	(ID: 494230452)
   6.	Февраль 2025	(ID: 734318053)
   7.	Январь 2025	(ID: 942372074)
   8.	Декабрь 2024	(ID: 731910267)
   9.	Ноябрь 2024	(ID: 2124199273)
  10.	Октябрь 2024	(ID: 791951036)
  11.	Сентябрь 2024	(ID: 777540678)
  12.	Август 2024	(ID: 1889846920)
  13.	Июль 2024	(ID: 562321949)
  14.	Июнь 2024	(ID: 2143264070)
  15.	Май 2024	(ID: 1354857949)
  16.	Апрель 2024	(ID: 905716784)
  17.	Март 2024	(ID: 2145965963)
  18.	Февраль 2024	(ID: 0)
  19.	Январь 2024	(ID: 1265629238)


In [6]:
# Словарь синонимов для нормализации названий колонок
COLUMN_SYNONYMS = {
    "Место": "rank",
    "Пред. месяц": "previous_rank",
    "Динамика": "rank_change",
    "Канал": "channel_name",
    '"Индекс Колезева"': "kolesev_index",
    
    "Total": "kolesev_index",
    "Итог по просмотрам*": "kolesev_index",

    "Прирост подписчиков": "subscriber_growth",
    "Просмотры видео+стримов": "video_views",
    "Просмотры шортсов": "shorts_views",
    "Самое популярное видео": "top_video_title",
    "Топовое видео": "top_video_title",
    "Популярное видео": "top_video_title",
    "Rank": "rank_technical",
    "Rank": "rank_technical",
    
    "Все видео": "total_videos",
    "Лайки всех видео": "total_likes",
    "Комменты ко всем видео": "total_comments",
    "Рост подписок": "subscriber_growth",
}


def normalize_column_name(column_name: str) -> str:
    """Нормализует название колонки, объединяя синонимы"""
    return COLUMN_SYNONYMS.get(column_name.strip(), column_name)

In [7]:
# Функция для анализа структуры колонок с нормализацией
async def analyze_sheet_structure(sheet_title: str) -> List[ColumnInfo]:
    """Анализирует структуру колонок на вкладке с нормализацией названий"""
    data = await parser.get_sheet_data(sheet_title)
    if not data:
        return []
        
    columns = []
    headers = data[0] if data else []
    
    for i, header in enumerate(headers):
        if header:  # Пропускаем пустые заголовки
            # Нормализуем название колонки
            normalized_name = normalize_column_name(header)
            
            # Собираем несколько значений для анализа типа данных
            sample_values = []
            for row_idx in range(1, min(4, len(data))):  # Берем до 3 строк для анализа
                if i < len(data[row_idx]):
                    sample_values.append(data[row_idx][i])
            
            column_info = ColumnInfo(
                name=normalized_name,  # Используем нормализованное название
                sheet_title=sheet_title,
                index=i,
                sample_values=sample_values
            )
            columns.append(column_info)
            
            
    print(f"🔍 Найдено {len(columns)} колонок на вкладке {sheet_title}")
    return columns

print("✅ Функция анализа структуры обновлена с нормализацией")

✅ Функция анализа структуры обновлена с нормализацией


In [8]:
# Функция консолидации данных с нормализацией заголовков
async def consolidate_all_sheets(exclude_sheets: Optional[List[str]] = None) -> ConsolidatedData:
    """Консолидирует данные со всех вкладок"""
    # Получаем список вкладок
    sheets = await parser.get_sheets_list(exclude_sheets)
    if not sheets:
        return ConsolidatedData(columns=[], data=[], source_sheets=[], total_rows=0)
    
    # Анализируем структуру всех вкладок
    all_columns: Set[str] = set()
    sheet_columns: Dict[str, List[ColumnInfo]] = {}
    
    print("🔄 Анализируем структуру вкладок...")
    for sheet in sheets:
        columns = await analyze_sheet_structure(sheet.title)
        sheet_columns[sheet.title] = columns
        for col in columns:
            all_columns.add(col.name)  # col.name уже нормализован
    
    # Создаем унифицированную схему
    unified_columns = sorted(list(all_columns))
    print(f"�� Унифицированная схема: {len(unified_columns)} колонок")
    
    # Консолидируем данные
    consolidated_data = []
    source_sheets = []
    
    for sheet in sheets:
        print(f"\n�� Обрабатываем вкладку: {sheet.title}")
        sheet_data = await parser.get_sheet_data(sheet.title)
        if not sheet_data:
            continue
            
        # Нормализуем заголовки
        raw_headers = sheet_data[0] if sheet_data else []
        normalized_headers = [normalize_column_name(header) for header in raw_headers]
        source_sheets.append(sheet.title)
        
        # Обрабатываем каждую строку данных
        for row_idx in range(1, len(sheet_data)):
            row_data = sheet_data[row_idx]
            consolidated_row = {}
            
            # Заполняем все колонки
            for col_name in unified_columns:
                col_value = None
                
                # Ищем колонку в текущей вкладке по нормализованному названию
                for i, header in enumerate(normalized_headers):
                    if header == col_name:
                        if i < len(row_data):
                            col_value = row_data[i]
                        break
                
                consolidated_row[col_name] = col_value
            
            # Добавляем информацию об источнике
            consolidated_row['_source_sheet'] = sheet.title
            consolidated_row['_row_number'] = row_idx
            
            consolidated_data.append(consolidated_row)
        
        print(f"  ✅ Обработано {len(sheet_data)-1} строк")
    
    print(f"\n🎉 Консолидировано {len(consolidated_data)} строк из {len(source_sheets)} вкладок")
    
    return ConsolidatedData(
        columns=unified_columns,
        data=consolidated_data,
        source_sheets=source_sheets,
        total_rows=len(consolidated_data)
    )

print("✅ Функция консолидации обновлена с нормализацией заголовков")

✅ Функция консолидации обновлена с нормализацией заголовков


In [ ]:
# Шаг 3: Консолидируем данные со всех вкладок
print("🔄 Шаг 3: Консолидируем данные со всех вкладок...")
consolidated_data = await consolidate_all_sheets(EXCLUDE_SHEETS)

if consolidated_data.total_rows > 0:
    print(f"\n✅ Консолидация завершена!")
    print(f"📊 Всего строк: {consolidated_data.total_rows}")
    print(f"🔢 Всего колонок: {len(consolidated_data.columns)}")
    print(f"📋 Источники: {', '.join(consolidated_data.source_sheets)}")
else:
    print("❌ Не удалось консолидировать данные")
    
df = pd.DataFrame(consolidated_data.data)
print(f"Размер: {df.shape}")
print(df.count())

# Сохраняем DataFrame в CSV файл
print("�� Сохраняем данные в CSV файл...")

# Создаем имя файла с текущей датой
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"youtube_ratings_consolidated_{timestamp}.csv"

# Сохраняем в CSV
df.to_csv(filename, index=False, encoding='utf-8-sig', sep='\t')


print(f"✅ Данные сохранены в файл: {filename}")
print(f"�� Размер файла: {df.shape[0]} строк × {df.shape[1]} колонок")


🔄 Шаг 3: Консолидируем данные со всех вкладок...
🚫 Исключена вкладка: ! Пояснения
📊 Всего найдено вкладок: 20
🔄 Вкладок для обработки: 19
🔄 Анализируем структуру вкладок...
📥 Получено 264 строк с вкладки Июль 2025
🔍 Найдено 10 колонок на вкладке Июль 2025
📥 Получено 264 строк с вкладки Июнь 2025
🔍 Найдено 10 колонок на вкладке Июнь 2025
📥 Получено 264 строк с вкладки Май 2025
🔍 Найдено 10 колонок на вкладке Май 2025
📥 Получено 230 строк с вкладки Апрель 2025
🔍 Найдено 10 колонок на вкладке Апрель 2025
📥 Получено 229 строк с вкладки Март 2025
🔍 Найдено 10 колонок на вкладке Март 2025
📥 Получено 229 строк с вкладки Февраль 2025
🔍 Найдено 10 колонок на вкладке Февраль 2025
📥 Получено 226 строк с вкладки Январь 2025
🔍 Найдено 10 колонок на вкладке Январь 2025
📥 Получено 223 строк с вкладки Декабрь 2024
🔍 Найдено 10 колонок на вкладке Декабрь 2024
📥 Получено 219 строк с вкладки Ноябрь 2024
🔍 Найдено 10 колонок на вкладке Ноябрь 2024
📥 Получено 220 строк с вкладки Октябрь 2024
🔍 Найдено 10 к

In [ ]:
# Шаг 1: Анализируем текущие типы данных
print("📊 Текущие типы данных:")
print(df.dtypes)
print("\n🔍 Уникальные значения в _source_sheet:")
print(df['_source_sheet'].unique())
print("\n🔍 Примеры kolesev_index:")
print(df['kolesev_index'].head(10))

# Шаг 2: Создаем функцию для парсинга русских дат
def parse_russian_date(date_str):
    """Парсит русскую дату вида 'Июль 2025' в datetime"""
    month_map = {
        'Январь': 1, 'Февраль': 2, 'Март': 3, 'Апрель': 4,
        'Май': 5, 'Июнь': 6, 'Июль': 7, 'Август': 8,
        'Сентябрь': 9, 'Октябрь': 10, 'Ноябрь': 11, 'Декабрь': 12
    }
    
    month, year = date_str.split()
    month_num = month_map[month]
    return pd.to_datetime(f"{year}-{month_num:02d}-01")

# Шаг 3: Очищаем kolesev_index
def clean_kolesev_index(value):
    """Очищает kolesev_index от пробелов и запятых"""
    if pd.isna(value) or value == 0:
        return 0
    # Убираем пробелы, заменяем запятую на точку
    cleaned = str(value).replace(' ', '').replace(',', '.')
    try:
        return int(float(cleaned))
    except:
        return pd.NA  # Возвращаем pd.NA для ошибок

# Шаг 4: Применяем преобразования
df_clean = df.copy()
df_clean['date'] = df_clean['_source_sheet'].apply(parse_russian_date)
df_clean['kolesev_index_clean'] = df_clean['kolesev_index'].apply(clean_kolesev_index).astype('Int64')

# Шаг 5: Проверяем результат
print("\n✅ После очистки:")
print(f"Даты: {df_clean['date'].min()} - {df_clean['date'].max()}")
print(f"kolesev_index: {df_clean['kolesev_index_clean'].describe()}")

df_clean.to_csv('youtube_ratings_data.csv', encoding='utf-8-sig', sep='\t')

📊 Текущие типы данных:
channel_name         object
kolesev_index        object
previous_rank        object
rank                 object
rank_change          object
rank_technical       object
shorts_views         object
subscriber_growth    object
top_video_title      object
total_comments       object
total_likes          object
total_videos         object
video_views          object
_source_sheet        object
_row_number           int64
dtype: object

🔍 Уникальные значения в _source_sheet:
['Июль 2025' 'Июнь 2025' 'Май 2025' 'Апрель 2025' 'Март 2025'
 'Февраль 2025' 'Январь 2025' 'Декабрь 2024' 'Ноябрь 2024' 'Октябрь 2024'
 'Сентябрь 2024' 'Июль 2024' 'Июнь 2024' 'Май 2024' 'Апрель 2024'
 'Март 2024' 'Февраль 2024' 'Январь 2024']

🔍 Примеры kolesev_index:
0    73 433 369,00
1     42 789 147,0
2     39 323 617,0
3     39 159 897,0
4     31 199 829,0
5     27 244 069,0
6     26 164 655,0
7     25 438 793,0
8     20 993 208,0
9     19 469 650,0
Name: kolesev_index, dtype: object

✅ Посл

In [24]:
# Шаг 6: Создаем pivot table для bar chart race
print("�� Создаем данные для bar chart race...")

# Сортируем по дате
df_clean = df_clean.sort_values('date')

# Создаем pivot table: каналы по строкам, даты по колонкам, значения - kolesev_index
pivot_data = df_clean.pivot_table(
    index='channel_name',
    columns='date',
    values='kolesev_index_clean',
    aggfunc='first'  # Берем первое значение если есть дубли
).fillna(0)

print(f"�� Размер pivot table: {pivot_data.shape}")
print(f"📅 Диапазон дат: {pivot_data.columns.min()} - {pivot_data.columns.max()}")
print(f"�� Количество каналов: {len(pivot_data)}")

# Шаг 7: Проверяем данные
print("\n🔍 Топ-5 каналов по последней дате:")
top_channels = pivot_data.iloc[:, -1].sort_values(ascending=False).head()
print(top_channels)

print("\n🔍 Пример данных (первые 5 каналов, последние 5 дат):")
print(pivot_data.iloc[:5, -5:])

# Шаг 8: Сохраняем подготовленные данные
pivot_data.to_csv('youtube_ratings_pivot.csv', encoding='utf-8-sig', sep='\t')
print("\n✅ Pivot table сохранен в 'youtube_ratings_pivot.csv'")

�� Создаем данные для bar chart race...
�� Размер pivot table: (295, 18)
📅 Диапазон дат: 2024-01-01 00:00:00 - 2025-07-01 00:00:00
�� Количество каналов: 295

🔍 Топ-5 каналов по последней дате:
channel_name
NEXTA Live           73433369.0
Ходорковский LIVE    42789147.0
Телеканал Дождь      39323617.0
Майкл Наки           39159897.0
Анатолий Шарий       31199829.0
Name: 2025-07-01 00:00:00, dtype: float64

🔍 Пример данных (первые 5 каналов, последние 5 дат):
date                                   2025-03-01  2025-04-01  2025-05-01  \
channel_name                                                                
                                              0.0         0.0         0.0   
7x7 Горизонтальная Россия                708725.0    275638.0    278642.0   
Alexandr Plushev                              0.0         0.0         0.0   
Alexey Arestovych                      14506077.0   9971172.0  10049312.0   
Andrei Illarionov (Андрей Илларионов)    852201.0    468370.0    303188.0  

In [35]:
# Создаем bar chart race
import bar_chart_race as bcr


def format_russian_date(date):
    """Форматирует дату в русский формат Месяц Год"""
    month_map = {
        1: 'Январь', 2: 'Февраль', 3: 'Март', 4: 'Апрель',
        5: 'Май', 6: 'Июнь', 7: 'Июль', 8: 'Август',
        9: 'Сентябрь', 10: 'Октябрь', 11: 'Ноябрь', 12: 'Декабрь'
    }
    return f"{month_map[date.month]} {date.year}"

def summary(values, ranks):
    total_views = int(round(values.sum(), -2))
    s = f'Total views - {total_views:,.0f}'
    return {'x': .99, 'y': .05, 's': s, 'ha': 'right', 'size': 8}



print("🎬 Создаем bar chart race...")

filename = f"ytr_race_top20.gif"
# Берем топ-20 каналов для анимации
top_n = 20
pivot_top = pivot_data.nlargest(top_n, pivot_data.columns[-1])
# Транспонируем данные: даты по строкам, каналы по колонкам
pivot_top = pivot_top.T
# Применяем к индексу
pivot_top.index = pivot_top.index.map(format_russian_date)




# Создаем анимацию
bcr.bar_chart_race(
    df=pivot_top,
    filename=filename,
    title='YouTube Ratings - Топ 20 каналов по Индексу Колезева',
    n_bars=top_n,
    steps_per_period=60,
    period_length=3000,
    figsize=(16, 9),  # Увеличиваем размер для лучшей читаемости
    cmap='tab20',      # 20 разных цветов для 20 каналов
    filter_column_colors=True,
    # Настройки для лучшей читаемости
    label_bars=True,   # Показывать значения на барах
    bar_size=0.8,      # Размер баров
    # Настройки шрифтов
    title_size=16,
    bar_label_size=10,
    tick_label_size=12,
    # Форматирование дат
    # period_fmt='%B %Y',  # Месяц Год (например: Июль 2025)
    perpendicular_bar_func='mean',
    period_summary_func=summary,
    fixed_max=True
)

print(f"✅ Анимация сохранена в '{filename}'")

🎬 Создаем bar chart race...


c:\Users\eremi\Documents\4. Projects\2024-07 YTRatings\yt_fetcher\.venv\Lib\site-packages\bar_chart_race\_make_chart.py:889: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_values.iloc[:, 0] = df_values.iloc[:, 0].fillna(method='ffill')
c:\Users\eremi\Documents\4. Projects\2024-07 YTRatings\yt_fetcher\.venv\Lib\site-packages\bar_chart_race\_make_chart.py:286: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(self.df_values.columns)
c:\Users\eremi\Documents\4. Projects\2024-07 YTRatings\yt_fetcher\.venv\Lib\site-packages\bar_chart_race\_make_chart.py:287: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([max_val] * len(ax.get_xticks()))
MovieWriter imagemagick unavailable; using Pillow instead.


✅ Анимация сохранена в 'ytr_race_top20.gif'
